# 🔬 Quantization Benchmark v4

Google Colab **T4 GPU** (16 GB VRAM).  
OOM-safe: `no_grad`, `detach`, per-layer cleanup, `use_cache=False`.

**Методы:** Uniform, GPTQ, AWQ, SmoothQuant, QuIP, WaterSIC, TurboQuant

In [ ]:
!pip install -q torch transformers datasets accelerate scipy tqdm matplotlib sentencepiece protobuf

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time, gc, copy
from typing import Dict, Tuple, Optional, List
from dataclasses import dataclass, field
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

# ═══════════════════════════════════════════
# Conv1D support (GPT-2)
# ═══════════════════════════════════════════
try:
    from transformers.pytorch_utils import Conv1D as HFConv1D
except ImportError:
    try:
        from transformers.modeling_utils import Conv1D as HFConv1D
    except ImportError:
        HFConv1D = None

LINEAR_TYPES = (nn.Linear, HFConv1D) if HFConv1D else (nn.Linear,)
print(f"Linear types: {[t.__name__ for t in LINEAR_TYPES]}")

torch.manual_seed(42)
np.random.seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
    vram = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"VRAM: {vram:.1f} GB")

In [ ]:
# ═══════════════════════════════════════════
# Conv1D ↔ Linear weight helpers
# ═══════════════════════════════════════════

def _is_conv1d(mod):
    return HFConv1D is not None and isinstance(mod, HFConv1D)


def get_weight(mod):
    """→ [d_out, d_in] float, detached, no grad."""
    W = mod.weight.detach().float()
    if _is_conv1d(mod):
        W = W.T                       # Conv1D: [d_in,d_out] → [d_out,d_in]
    return W


def set_weight(mod, W_deq):
    """Записывает [d_out, d_in] обратно, requires_grad=False."""
    W = W_deq.detach()
    if _is_conv1d(mod):
        W = W.T                       # → [d_in,d_out]
    mod.weight = nn.Parameter(W.to(mod.weight.dtype), requires_grad=False)

---
## 1. Модель и данные

In [ ]:
MODEL_NAME = "openai-community/gpt2"
# MODEL_NAME = "facebook/opt-125m"
# MODEL_NAME = "EleutherAI/pythia-160m"

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model_fp = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float32,
).to(DEVICE).eval()

# Отключаем градиенты у всех параметров (экономия VRAM)
for p in model_fp.parameters():
    p.requires_grad_(False)

n_params = sum(p.numel() for p in model_fp.parameters())
tc = {}
for n, m in model_fp.named_modules():
    if isinstance(m, LINEAR_TYPES):
        tc[type(m).__name__] = tc.get(type(m).__name__, 0) + 1

print(f"Loaded: {n_params/1e6:.1f}M params")
print(f"Layer types: {tc}")
if DEVICE == "cuda":
    print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
def load_wikitext2(tokenizer):
    ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
    return tokenizer("\n\n".join(ds["text"]), return_tensors="pt").input_ids


def load_calibration_data(tokenizer, n_samples=64, seq_len=1024):
    ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
    texts = [t for t in ds["text"] if len(t.strip()) > 200]
    enc = tokenizer(
        "\n\n".join(texts), return_tensors="pt",
        truncation=True, max_length=seq_len * n_samples * 2,
    )
    ids = enc.input_ids[0]
    out = []
    for i in range(0, len(ids) - seq_len, seq_len):
        out.append(ids[i : i + seq_len])
        if len(out) >= n_samples:
            break
    return torch.stack(out)


def load_lambada(tokenizer, n_samples=200):
    ds = load_dataset("EleutherAI/lambada_openai", "en", split="test")
    return [x["text"] for x in ds.select(range(min(n_samples, len(ds))))]


SEQ_LEN = min(1024, getattr(model_fp.config, "max_position_embeddings", 1024))

print("Loading datasets...")
wikitext_ids = load_wikitext2(tokenizer)
calib_data = load_calibration_data(tokenizer, n_samples=64, seq_len=SEQ_LEN)
lambada_samples = load_lambada(tokenizer, 200)
print(f"  WikiText-2: {wikitext_ids.shape[1]} tok | Calib: {calib_data.shape} | LAMBADA: {len(lambada_samples)}")

---
## 2. Evaluation (OOM-safe)

In [ ]:
@torch.no_grad()
def eval_perplexity(model, input_ids, device="cuda",
                    max_length=None, stride=512):
    if max_length is None:
        max_length = min(
            getattr(model.config, "max_position_embeddings", 1024), 1024
        )
    seq_len = input_ids.size(1)
    nlls, n_tokens, prev_end = [], 0, 0

    for begin in tqdm(range(0, seq_len, stride), desc="PPL", leave=False):
        end = min(begin + max_length, seq_len)
        trg_len = end - prev_end
        chunk = input_ids[:, begin:end].to(device)
        target = chunk.clone()
        target[:, :-trg_len] = -100

        # use_cache=False — не накапливаем KV-cache
        loss = model(chunk, labels=target, use_cache=False).loss
        actual = (target != -100).sum().item()
        nlls.append(loss.float() * actual)
        n_tokens += actual
        prev_end = end
        if end == seq_len:
            break

    return torch.exp(torch.stack(nlls).sum() / n_tokens).item()


@torch.no_grad()
def eval_lambada(model, tokenizer, samples, device="cuda"):
    correct = total = 0
    for text in tqdm(samples, desc="LAMBADA", leave=False):
        parts = text.rsplit(" ", 1)
        if len(parts) < 2:
            continue
        target_ids = tokenizer(" " + parts[1], return_tensors="pt").input_ids[0]
        full_ids = tokenizer(text, return_tensors="pt").input_ids.to(device)
        logits = model(full_ids, use_cache=False).logits[0]
        n_t = len(target_ids)
        if torch.equal(logits[-(n_t+1):-1].argmax(-1), target_ids.to(device)):
            correct += 1
        total += 1
    return correct / max(total, 1)


def eval_weight_metrics(model_orig, model_quant):
    mods_o = dict(model_orig.named_modules())
    mods_q = dict(model_quant.named_modules())
    per_layer = []
    for name, mo in mods_o.items():
        if not isinstance(mo, LINEAR_TYPES):
            continue
        mq = mods_q.get(name)
        if mq is None:
            continue
        wo = get_weight(mo)
        wq = get_weight(mq)
        mse = F.mse_loss(wo, wq).item()
        cos = F.cosine_similarity(wo.flatten()[None], wq.flatten()[None]).item()
        snr = 10 * np.log10((wo**2).mean().item() / (mse + 1e-20))
        per_layer.append(dict(name=name, mse=mse, cos_sim=cos, snr_db=snr))
    if not per_layer:
        return dict(per_layer=[], avg_mse=0, avg_cos_sim=1, avg_snr_db=999)
    return dict(
        per_layer=per_layer,
        avg_mse=np.mean([m["mse"] for m in per_layer]),
        avg_cos_sim=np.mean([m["cos_sim"] for m in per_layer]),
        avg_snr_db=np.mean([m["snr_db"] for m in per_layer]),
    )

In [ ]:
print("FP32 baseline...")
ppl_fp32 = eval_perplexity(model_fp, wikitext_ids, DEVICE)
acc_fp32 = eval_lambada(model_fp, tokenizer, lambada_samples, DEVICE)
print(f"  PPL={ppl_fp32:.2f}  LAMBADA={acc_fp32*100:.1f}%")

---
## 3. Инфраструктура квантования (OOM-safe)

In [ ]:
# ═══════════════════════════════════════════
# Сбор активаций — всё под no_grad
# ═══════════════════════════════════════════

@torch.no_grad()
def collect_layer_inputs(model, calib_data, device="cuda", n_samples=32):
    hooks, store = [], {}

    def _hook(name):
        def fn(_, inp, __):
            x = inp[0].detach().float().cpu()
            if x.dim() == 3:
                x = x.reshape(-1, x.shape[-1])
            store.setdefault(name, []).append(x)
        return fn

    for name, mod in model.named_modules():
        if isinstance(mod, LINEAR_TYPES):
            hooks.append(mod.register_forward_hook(_hook(name)))

    for i in tqdm(range(min(n_samples, len(calib_data))),
                  desc="Activations", leave=False):
        model(calib_data[i].unsqueeze(0).to(device), use_cache=False)

    for h in hooks:
        h.remove()

    out = {k: torch.cat(v, 0) for k, v in store.items()}
    print(f"  Collected {len(out)} layers")
    return out

In [ ]:
# ═══════════════════════════════════════════
# Утилиты квантования
# ═══════════════════════════════════════════

def compute_hessian(X):
    """X: [n, d] → H: [d, d]"""
    n = X.shape[0]
    H = (X.T @ X).float() / n
    damp = 0.01 * H.diagonal().mean()
    H.diagonal().add_(damp)
    return H


def random_orthogonal(d, device="cpu"):
    Q, R = torch.linalg.qr(torch.randn(d, d, device=device))
    return Q @ torch.diag(R.diagonal().sign())


def fake_quant_sym(W, bits, dim=1):
    qmax = 2 ** (bits - 1) - 1
    scale = W.abs().amax(dim=dim, keepdim=True) / qmax
    scale = scale.clamp(min=1e-10)
    return (W / scale).round().clamp(-qmax, qmax) * scale

In [ ]:
# ═══════════════════════════════════════════
# Оркестратор — OOM-safe
#
# 1. @torch.no_grad() на весь цикл
# 2. get_weight() возвращает detached tensor
# 3. set_weight() записывает Parameter(requires_grad=False)
# 4. del + empty_cache после каждого слоя
# ═══════════════════════════════════════════

SKIP = ["lm_head", "embed", "wte", "wpe",
        "ln_", "layernorm", "layer_norm", ".norm"]


def _skip(name):
    nl = name.lower()
    return any(p in nl for p in SKIP)


@torch.no_grad()     # ← КРИТИЧНО: отключает граф вычислений
def quantize_model(model_fp, quantize_fn, layer_inputs=None,
                   bits=4, device="cuda"):
    """quantize_fn(W, X, bits) → W_deq.  W: [d_out, d_in]."""

    model_q = copy.deepcopy(model_fp).cpu()

    # Все параметры — requires_grad=False
    for p in model_q.parameters():
        p.requires_grad_(False)

    n_q = n_s = 0

    for name, mod in tqdm(list(model_q.named_modules()),
                          desc="Quantizing", leave=False):
        if not isinstance(mod, LINEAR_TYPES):
            continue
        if _skip(name):
            n_s += 1
            continue

        # detach: оригинальный граф не протечёт
        W = get_weight(mod)                    # [d_out, d_in], detached
        X = layer_inputs.get(name) if layer_inputs else None

        try:
            W_deq = quantize_fn(W, X, bits)    # всё на CPU, без градиентов
            set_weight(mod, W_deq)             # Parameter(requires_grad=False)
            n_q += 1
        except Exception as e:
            print(f"  ⚠ {name}: {e}")
            n_s += 1

        # Очистка: не даём мусору копиться
        del W
        if X is not None:
            del X

    gc.collect()
    print(f"  quantized {n_q}, skipped {n_s}")
    model_q = model_q.to(device).eval()

    if device == "cuda":
        torch.cuda.empty_cache()

    return model_q


@dataclass
class Result:
    name: str
    bits: int
    ppl: float
    lambada_acc: float
    avg_mse: float
    avg_cos_sim: float
    avg_snr_db: float
    quant_time: float
    per_layer: Optional[List] = None


def run_benchmark(tag, quantize_fn, model_fp, bits,
                  wikitext_ids, lambada_samples, tokenizer,
                  layer_inputs=None, device="cuda"):
    print(f"\n{'─'*55}")
    print(f"▶ {tag}")

    t0 = time.time()
    model_q = quantize_model(
        model_fp, quantize_fn, layer_inputs, bits, device
    )
    qt = time.time() - t0

    wm = eval_weight_metrics(model_fp, model_q)
    ppl = eval_perplexity(model_q, wikitext_ids, device=device)
    acc = eval_lambada(model_q, tokenizer, lambada_samples, device=device)

    print(f"  PPL={ppl:.2f}  ΔPPL={ppl-ppl_fp32:+.2f}  "
          f"Acc={acc*100:.1f}%  SNR={wm['avg_snr_db']:.1f}dB  "
          f"CosSim={wm['avg_cos_sim']:.5f}  Time={qt:.1f}s")

    del model_q
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

    return Result(
        tag, bits, ppl, acc,
        wm["avg_mse"], wm["avg_cos_sim"], wm["avg_snr_db"],
        qt, wm["per_layer"],
    )


print("Pipeline loaded ✓")

---
## 4. Методы квантования

`fn(W, X, bits) → W_deq`. W всегда `[d_out, d_in]`, detached.

### 4.1  Uniform / RTN

In [ ]:
def quantize_uniform(W, X, bits):
    return fake_quant_sym(W, bits, dim=1)

### 4.2  GPTQ

Per-row scale. H_inv на CPU. Никаких графов.

In [ ]:
def quantize_gptq(W, X, bits):
    if X is None:
        return fake_quant_sym(W, bits, dim=1)

    d_out, d_in = W.shape
    qmax = 2 ** (bits - 1) - 1

    H = compute_hessian(X)
    H_inv = torch.cholesky_inverse(torch.linalg.cholesky(H))

    row_scale = (W.abs().amax(dim=1) / qmax).clamp(min=1e-10)  # [d_out]

    W = W.clone()
    for j in range(d_in):
        w_j = W[:, j]
        q_j = (w_j / row_scale).round().clamp(-qmax, qmax) * row_scale
        err_j = w_j - q_j
        W[:, j] = q_j

        if j < d_in - 1:
            coeff = H_inv[j, j+1:] / (H_inv[j, j] + 1e-12)
            W[:, j+1:] -= err_j.unsqueeze(1) @ coeff.unsqueeze(0)

    return W

### 4.3  AWQ

`WX_orig` вынесен из цикла.

In [ ]:
def quantize_awq(W, X, bits):
    if X is None:
        return fake_quant_sym(W, bits, dim=1)

    s = X.abs().mean(dim=0)
    X_sub = X[:min(512, X.shape[0])].T
    WX_orig = W @ X_sub                          # precompute

    best_loss, best_W = float("inf"), None
    for alpha in torch.linspace(0, 1, 20):
        cs = s.pow(alpha.item()).clamp(min=1e-8)
        Ws = W * cs.unsqueeze(0)
        Wq = fake_quant_sym(Ws, bits, dim=1)
        Wd = Wq / cs.unsqueeze(0)
        loss = (WX_orig - Wd @ X_sub).pow(2).sum().item()
        if loss < best_loss:
            best_loss, best_W = loss, Wd

    return best_W

### 4.4  SmoothQuant (W8A8)

Activation quant per-token. W8A8 error считается внутри.

In [ ]:
_sq_errors = []

def quantize_smoothquant(W, X, bits, alpha=0.5):
    if X is None:
        return fake_quant_sym(W, bits, dim=1)

    act_max = X.abs().amax(dim=0).clamp(min=1e-8)
    w_max = W.abs().amax(dim=0).clamp(min=1e-8)
    s = (act_max.pow(alpha) / w_max.pow(1 - alpha)).clamp(min=1e-8)

    W_sm = W * s.unsqueeze(0)
    X_sm = X / s.unsqueeze(0)

    W_sq = fake_quant_sym(W_sm, bits, dim=1)
    X_sq = fake_quant_sym(X_sm, bits, dim=-1)  # per-token

    n_sub = min(512, X.shape[0])
    Y_fp = X_sm[:n_sub] @ W_sm.T
    Y_q  = X_sq[:n_sub] @ W_sq.T
    _sq_errors.append((Y_fp - Y_q).norm().item() / (Y_fp.norm().item() + 1e-12))

    return W_sq / s.unsqueeze(0)

### 4.5  QuIP

In [ ]:
def quantize_quip(W, X, bits):
    d_out, d_in = W.shape
    U = random_orthogonal(d_out, W.device)
    V = random_orthogonal(d_in, W.device)
    Wr = U @ W @ V
    Wrq = fake_quant_sym(Wr, bits, dim=1)
    return U.T @ Wrq @ V.T

### 4.6  WaterSIC ⭐

In [ ]:
def quantize_watersic(W, X, bits):
    if X is None:
        return fake_quant_sym(W, bits, dim=1)

    d_out, d_in = W.shape
    H = compute_hessian(X)
    H_inv = torch.cholesky_inverse(torch.linalg.cholesky(H))

    diag_H = H.diagonal().clamp(min=1e-12)
    log_d = torch.log2(diag_H)
    R = float(bits) + 0.5 * (log_d - log_d.mean())
    R = R.clamp(min=2.0, max=8.0)
    R = R * (float(bits) * d_in) / R.sum()

    W = W.clone()
    for j in range(d_in):
        qmax_j = max(int(2 ** R[j].item()) // 2 - 1, 1)
        w_j = W[:, j]
        sc = (w_j.abs().max() / max(qmax_j, 1)).clamp(min=1e-10)
        q_j = (w_j / sc).round().clamp(-qmax_j, qmax_j) * sc
        err_j = w_j - q_j
        W[:, j] = q_j
        if j < d_in - 1:
            coeff = H_inv[j, j+1:] / (H_inv[j, j] + 1e-12)
            W[:, j+1:] -= err_j.unsqueeze(1) @ coeff.unsqueeze(0)

    return W

### 4.7  TurboQuant ⭐ (KV-cache)

In [ ]:
def turboquant_vectors(V, bits=3.5, use_qjl=True):
    n, d = V.shape
    dev = V.device
    R = random_orthogonal(d, dev)
    Vr = V @ R.T
    norms = Vr.norm(dim=-1, keepdim=True).clamp(min=1e-12)
    U = Vr / norms
    signs = U.sign()
    qmax = 2 ** int(bits) - 1
    Uq = (U.abs() * qmax).round().clamp(0, qmax) / qmax
    Vhr = signs * Uq * norms

    if use_qjl:
        res = Vr - Vhr
        m = d
        S = torch.randn(m, d, device=dev).sign() / (m ** 0.5)
        sk = (S @ res.T).sign()
        rs = res.norm(dim=-1).mean() / (m ** 0.5)
        Vhr = Vhr + rs * (S.T @ sk).T

    Vh = Vhr @ R
    return Vh, {
        "cos_sim": F.cosine_similarity(V.flatten()[None], Vh.flatten()[None]).item(),
        "mse": F.mse_loss(V, Vh).item(),
    }


@torch.no_grad()
def eval_turboquant(model, wikitext_ids, bits_list=(4., 3.5, 3., 2.5),
                    device="cuda", max_chunks=15):
    ml = min(getattr(model.config, "max_position_embeddings", 1024), 512)
    seq_len = wikitext_ids.size(1)
    res = {b: {"cos": [], "mse": []} for b in bits_list}
    n = 0
    for begin in tqdm(range(0, seq_len, ml), desc="TQ", leave=False):
        end = min(begin + ml, seq_len)
        chunk = wikitext_ids[:, begin:end].to(device)
        out = model(chunk, output_hidden_states=True, use_cache=False)
        for h in out.hidden_states:
            V = h[0].float()
            if V.shape[0] < 8:
                continue
            for b in bits_list:
                _, info = turboquant_vectors(V, bits=b)
                res[b]["cos"].append(info["cos_sim"])
                res[b]["mse"].append(info["mse"])
        n += 1
        if n >= max_chunks:
            break
    return {
        b: {"cos_sim": np.mean(v["cos"]), "mse": np.mean(v["mse"])}
        for b, v in res.items() if v["cos"]
    }

---
## 5. Сбор активаций

In [ ]:
print("Collecting activations...")
layer_inputs = collect_layer_inputs(
    model_fp, calib_data, DEVICE, n_samples=min(32, len(calib_data))
)
for n, x in list(layer_inputs.items())[:3]:
    print(f"  {n}: {x.shape}")

if DEVICE == "cuda":
    print(f"VRAM after collection: {torch.cuda.memory_allocated()/1e9:.2f} GB")

---
## 6. 🏃 Запуск

In [ ]:
METHODS = [
    ("Uniform-8bit",      quantize_uniform,     8, False),
    ("Uniform-4bit",      quantize_uniform,     4, False),
    ("Uniform-3bit",      quantize_uniform,     3, False),
    ("GPTQ-4bit",         quantize_gptq,        4, True),
    ("GPTQ-3bit",         quantize_gptq,        3, True),
    ("AWQ-4bit",          quantize_awq,          4, True),
    ("AWQ-3bit",          quantize_awq,          3, True),
    ("SmoothQuant-8bit",  quantize_smoothquant,  8, True),
    ("QuIP-4bit",         quantize_quip,         4, False),
    ("QuIP-3bit",         quantize_quip,         3, False),
    ("WaterSIC-4bit",     quantize_watersic,     4, True),
    ("WaterSIC-3bit",     quantize_watersic,     3, True),
]

all_results = [
    Result("FP32", 32, ppl_fp32, acc_fp32, 0, 1.0, float("inf"), 0)
]
_sq_errors.clear()

for tag, fn, bits, needs_X in METHODS:
    try:
        r = run_benchmark(
            tag, fn, model_fp, bits,
            wikitext_ids, lambada_samples, tokenizer,
            layer_inputs if needs_X else None, DEVICE,
        )
        all_results.append(r)
    except Exception as e:
        print(f"  ✗ {tag}: {e}")
        import traceback; traceback.print_exc()
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

---
## 7. TurboQuant

In [ ]:
print("TurboQuant KV-cache eval...")
tq = eval_turboquant(model_fp, wikitext_ids, device=DEVICE, max_chunks=15)
print(f"{'Bits':>6}  {'CosSim':>10}  {'MSE':>12}")
for b in sorted(tq, reverse=True):
    print(f"{b:>6.1f}  {tq[b]['cos_sim']:>10.6f}  {tq[b]['mse']:>12.6f}")

---
## 8. SmoothQuant W8A8

In [ ]:
if _sq_errors:
    print(f"SmoothQuant W8A8 matmul error")
    print(f"  Avg  ||Y_fp−Y_q|| / ||Y_fp|| = {np.mean(_sq_errors):.6f}")
    print(f"  Max = {np.max(_sq_errors):.6f}  Min = {np.min(_sq_errors):.6f}")
    print(f"  Layers: {len(_sq_errors)}")

---
## 9. 📊 Сводка

In [ ]:
print(f"\n{'═'*90}")
print(f"{'RESULTS — ' + MODEL_NAME:^90}")
print(f"{'═'*90}")
print(f"{'Method':<22} {'Bits':>4} {'PPL':>8} {'ΔPPL':>7} "
      f"{'LAMBADA':>8} {'CosSim':>9} {'SNR(dB)':>8} {'Time':>6}")
print(f"{'─'*90}")
for r in all_results:
    dp = r.ppl - ppl_fp32
    sn = f"{r.avg_snr_db:.1f}" if r.avg_snr_db < 999 else "∞"
    print(f"{r.name:<22} {r.bits:>4} {r.ppl:>8.2f} {dp:>+7.2f} "
          f"{r.lambada_acc*100:>7.1f}% {r.avg_cos_sim:>9.6f} "
          f"{sn:>8} {r.quant_time:>5.1f}s")
print(f"{'═'*90}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
names = [r.name for r in all_results]
cmap = plt.cm.tab20(np.linspace(0, 1, len(names)))

def _bar(ax, vals, title, fmt=".1f", fs=6):
    bars = ax.bar(names, vals, color=cmap)
    ax.set_title(title)
    for b, v in zip(bars, vals):
        ax.text(b.get_x()+b.get_width()/2, b.get_height(),
                f"{v:{fmt}}", ha="center", va="bottom", fontsize=fs)
    ax.tick_params(axis="x", rotation=55, labelsize=6)

_bar(axes[0,0], [r.ppl for r in all_results], "PPL ↓")
_bar(axes[0,1], [r.ppl-ppl_fp32 for r in all_results], "ΔPPL ↓", "+.1f")
axes[0,1].axhline(0, color="k", lw=.5)
_bar(axes[0,2], [r.lambada_acc*100 for r in all_results], "LAMBADA % ↑")

qr = [r for r in all_results if r.name != "FP32"]
qn = [r.name for r in qr]
qc = plt.cm.tab20(np.linspace(0, 1, len(qn)))

ax = axes[1,0]
cv = [r.avg_cos_sim for r in qr]
bars = ax.bar(qn, cv, color=qc)
ax.set_title("CosSim ↑")
ax.set_ylim(min(cv)*0.999, 1.0002)
for b, v in zip(bars, cv):
    ax.text(b.get_x()+b.get_width()/2, b.get_height(), f"{v:.4f}", ha="center", va="bottom", fontsize=5)
ax.tick_params(axis="x", rotation=55, labelsize=6)

ax = axes[1,1]
sv = [r.avg_snr_db for r in qr]
bars = ax.bar(qn, sv, color=qc)
ax.set_title("SNR dB ↑")
for b, v in zip(bars, sv):
    ax.text(b.get_x()+b.get_width()/2, b.get_height(), f"{v:.1f}", ha="center", va="bottom", fontsize=6)
ax.tick_params(axis="x", rotation=55, labelsize=6)

ax = axes[1,2]
for r, c in zip(all_results, cmap):
    bv = r.bits if r.bits <= 8 else 16
    ax.scatter(bv, r.ppl, s=100, color=c, edgecolors="k", zorder=3)
    ax.annotate(r.name, (bv, r.ppl), textcoords="offset points", xytext=(4,4), fontsize=6)
ax.set_xlabel("Bits"); ax.set_ylabel("PPL"); ax.set_title("Pareto")
ax.grid(True, alpha=.3)

fig.suptitle(f"Quantization Benchmark — {MODEL_NAME}", fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# Per-layer SNR
show = [r for r in all_results if r.per_layer and r.bits <= 4]
if show:
    fig, axes = plt.subplots(len(show), 1, figsize=(14, 3.5*len(show)), squeeze=False)
    for i, r in enumerate(show):
        snrs = [m["snr_db"] for m in r.per_layer]
        axes[i,0].bar(range(len(snrs)), snrs, alpha=.7, width=1)
        axes[i,0].axhline(np.mean(snrs), color="r", ls="--", label=f"mean={np.mean(snrs):.1f}")
        axes[i,0].set_title(f"{r.name}"); axes[i,0].set_ylabel("SNR dB"); axes[i,0].legend(fontsize=8)
    plt.tight_layout(); plt.show()

In [ ]:
# TurboQuant graph
if tq:
    bv = sorted(tq); cv = [tq[b]["cos_sim"] for b in bv]; mv = [tq[b]["mse"] for b in bv]
    fig, (a1,a2) = plt.subplots(1,2,figsize=(12,4))
    a1.plot(bv, cv, "o-", color="teal", lw=2, ms=8); a1.set_xlabel("Bits"); a1.set_ylabel("CosSim")
    a1.set_title("TurboQuant CosSim"); a1.grid(True, alpha=.3)
    for b,c in zip(bv,cv): a1.annotate(f"{c:.5f}", (b,c), textcoords="offset points", xytext=(0,8), fontsize=9, ha="center")
    a2.plot(bv, mv, "s-", color="coral", lw=2, ms=8); a2.set_xlabel("Bits"); a2.set_ylabel("MSE")
    a2.set_title("TurboQuant MSE"); a2.grid(True, alpha=.3)
    for b,m in zip(bv,mv): a2.annotate(f"{m:.4f}", (b,m), textcoords="offset points", xytext=(0,8), fontsize=9, ha="center")
    plt.tight_layout(); plt.show()

---
## Changelog v4 — OOM fixes

```
PROBLEM: Kernel OOM crash на T4 (16 GB).
         PyTorch строил граф вычислений внутри квантования,
         накапливая промежуточные тензоры в VRAM.

FIX 1:  @torch.no_grad() на quantize_model()
        Полностью отключает граф. Ни одна операция внутри
        (H_inv, matmul, outer product) не создаёт grad tensors.

FIX 2:  get_weight() возвращает .detach().float()
        set_weight() записывает Parameter(requires_grad=False)
        Старый граф от model_fp не протекает в model_q.

FIX 3:  model_fp — все параметры requires_grad_(False)
        Экономит VRAM ещё на этапе загрузки.

FIX 4:  gc.collect() + torch.cuda.empty_cache() после каждого метода.
        Не даём мусору копиться между бенчмарками.

FIX 5:  use_cache=False в eval_perplexity / eval_lambada
        KV-cache не накапливается при проходе по WikiText-2.

FIX 6:  Квантование на CPU (model_q.cpu()), потом .to(device)
        GPTQ/WaterSIC цикл по столбцам не занимает GPU.
```